In [ ]:
import json
import os
import glob
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Make the project root importable so `from src.bots import ...` works
sys.path.insert(0, str(Path.cwd().parent))

## Load data files & Mouse trajectory preview

In [ ]:
from src.plotting import plot_trajectory
from src.data import find_red_eclipse_files, load_red_eclipse_mouse

game_files = find_red_eclipse_files()

for file_path in game_files[:5]:
    meta, mouse = load_red_eclipse_mouse(file_path)
    if mouse.empty:
        continue
    plot_trajectory(mouse, title=f"userId={meta['userId']}, gameId={meta['gameId']}")
    print(f"\nuserId={meta['userId']}, gameId={meta['gameId']}, events={len(mouse)}")


## Feature extraction


In [ ]:
from src.features import extract_features
from src.data import load_red_eclipse_mouse

game_rows = []
for file_path in game_files:
    meta, mouse = load_red_eclipse_mouse(file_path)
    features = extract_features(mouse)

    if features is None:
        continue

    features.update({
        **meta,
        "is_bot": 0,
        "bot_type": "human",
    })
    game_rows.append(features)

games_df = pd.DataFrame(game_rows)
games_df.to_csv("../data/red_eclipse_features.csv", index=False)
print(games_df.head())


## Player identification


In [ ]:
from src.features import feature_cols

MIN_GAMES = 8
games_df_37 = games_df.groupby("userId").filter(lambda g: len(g) >= MIN_GAMES)

# games_df = all 45 players, games_df_37 = more than 8 games
select_model = games_df_37

input_data = select_model[feature_cols]
output_data = select_model["userId"]

input_train, input_test, output_train, output_test = train_test_split(
    input_data, output_data, test_size=0.2, random_state=42, stratify=output_data
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(input_train, output_train)

output_pred = model.predict(input_test)
accuracy = accuracy_score(output_test, output_pred)

print(f"Accuracy: {accuracy:.2%}")
print(f"Random baseline: {1 / output_data.nunique():.2%} ({output_data.nunique()} players, {len(select_model)} games)")
print()
print(classification_report(output_test, output_pred))


## Block-bootstrap synthetic bots


In [ ]:
from src.bots import (
    build_segments,
    stitch_bot_game,
    collect_human_motion_samples,
    median_trace_duration_ms,
)
from src.features import extract_features
from src.config import RNG_SEED
from src.data import load_red_eclipse_mouse

N_BOT_GAMES = len(games_df)  # one bot game per human game

rng = np.random.default_rng(RNG_SEED)

# 1) Build segment pool + empirical motion stats from human games
segment_pool = []
re_human_mice = []
for file_path in game_files:
    meta, mouse = load_red_eclipse_mouse(file_path)
    re_human_mice.append(mouse)
    segment_pool.extend(build_segments(mouse, rng=rng))

re_motion = collect_human_motion_samples(re_human_mice, rng=rng)
re_dt_samples = re_motion["dt_samples"]
re_dt_by_session = re_motion["dt_by_session"]
re_target_ms = median_trace_duration_ms(re_human_mice)
print(
    f"Segment pool: {len(segment_pool)} segments from {len(game_files)} games | "
    f"dt n={len(re_dt_samples)} sessions={len(re_dt_by_session)} median={np.median(re_dt_samples):.2f}ms | "
    f"step median={np.median(re_motion['step_samples']):.3f} | "
    f"target={re_target_ms/1000:.1f}s"
)

# 2) Generate synthetic bot games
N_PREVIEW = 5              # save first N trajectories for render graph
bot_rows = []
sample_bot_trajectories = []
for i in range(N_BOT_GAMES):
    bot_mouse = stitch_bot_game(
        segment_pool,
        dt_samples=re_dt_samples,
        dt_by_session=re_dt_by_session,
        target_duration_ms=re_target_ms,
        rng=rng,
    )
    if len(sample_bot_trajectories) < N_PREVIEW:
        sample_bot_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({
        "userId": -1,
        "gameId": f"bot_{i}",
        "source_file": f"synthetic_bot_{i}",
        "is_bot": 1,
        "bot_type": "stitch",
    })
    bot_rows.append(feats)

bots_stitch_df = pd.DataFrame(bot_rows)
print(f"Generated {len(bots_stitch_df)} stitch bot games")
print(bots_stitch_df.head())



## Block-bootstrap synthetic bots trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_bot_trajectories):
    plot_trajectory(df, title=f"bot_{i}")
    print(f"\nbot_{i}, events={len(df)}")


## Smooth bot generation


In [ ]:
from src.bots import (
    estimate_smooth_params,
    generate_smooth_bot_game,
    smooth_params_for_print,
)
from src.features import extract_features

median_events = int(games_df["n_events"].median())
N_SMOOTH_BOTS = len(games_df)

re_smooth_params = estimate_smooth_params(games_df, **re_motion)
print(f"RE smooth params: {smooth_params_for_print(re_smooth_params)}")

smooth_rows = []
sample_smooth_trajectories = []
for i in range(N_SMOOTH_BOTS):
    bot_mouse = generate_smooth_bot_game(
        n_events=median_events, seed=RNG_SEED + i, **re_smooth_params
    )
    if len(sample_smooth_trajectories) < N_PREVIEW:
        sample_smooth_trajectories.append(bot_mouse.copy())

    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({
        "userId": -2,
        "gameId": f"smooth_{i}",
        "source_file": f"synthetic_smooth_{i}",
        "is_bot": 1,
        "bot_type": "smooth",
    })
    smooth_rows.append(feats)

bots_smooth_df = pd.DataFrame(smooth_rows)
print(f"Generated {len(bots_smooth_df)} smooth bot games (n_events={median_events})")
print(bots_smooth_df.head())


## Smooth bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_smooth_trajectories):
    plot_trajectory(df, title=f"smooth_{i}")
    print(f"\nsmooth_{i}, events={len(df)}")

## Bézier bot generation


In [ ]:
from src.bots import (
    estimate_bezier_params,
    generate_bezier_bot_game,
    bezier_params_for_print,
)
from src.features import extract_features

N_BEZIER_BOTS = len(games_df)

re_bezier_params = estimate_bezier_params(games_df, **re_motion)
print(f"RE bezier params: {bezier_params_for_print(re_bezier_params)}")

bezier_rows = []
sample_bezier_trajectories = []
for i in range(N_BEZIER_BOTS):
    bot_mouse = generate_bezier_bot_game(
        n_events=median_events, seed=RNG_SEED + 50 + i, **re_bezier_params
    )
    if len(sample_bezier_trajectories) < N_PREVIEW:
        sample_bezier_trajectories.append(bot_mouse.copy())

    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({
        "userId": -3,
        "gameId": f"bezier_{i}",
        "source_file": f"synthetic_bezier_{i}",
        "is_bot": 1,
        "bot_type": "bezier",
    })
    bezier_rows.append(feats)

bots_bezier_df = pd.DataFrame(bezier_rows)
print(f"Generated {len(bots_bezier_df)} bezier bot games (n_events={median_events})")
print(bots_bezier_df.head())



## Bézier bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_bezier_trajectories):
    plot_trajectory(df, title=f"bezier_{i}")
    print(f"\nbezier_{i}, events={len(df)}")



## VAE bot — train once / load weights (RE)

**Order:** train (or load `.pt`) → generate full games with `stitch_bot_game` → preview → (next) Human vs Bot.

Default: load `artifacts/vae_re_v1.pt`. Set `VAE_FORCE_RETRAIN = True` only if data/protocol changed.

In [ ]:
from pathlib import Path
import numpy as np

from src.config import PROJECT_ROOT, RNG_SEED
from src.vae_bot import ensure_vae_bundle, DEFAULT_RE_WEIGHTS

# Formal runs: False → load committed weights. True → retrain + overwrite path.
VAE_FORCE_RETRAIN = False
VAE_WEIGHTS_PATH = DEFAULT_RE_WEIGHTS  # artifacts/vae_re_v1.pt

re_step_median = float(np.median(re_motion["step_samples"]))
re_vae_bundle = ensure_vae_bundle(
    re_human_mice,
    re_step_median,
    path=VAE_WEIGHTS_PATH,
    force_retrain=VAE_FORCE_RETRAIN,
    seed=RNG_SEED,
)
print(
    f"RE VAE ready | path={Path(VAE_WEIGHTS_PATH)} | "
    f"seg_len={re_vae_bundle['seg_len']} z={re_vae_bundle['z_dim']} "
    f"step_median={re_vae_bundle['step_median']:.4f} "
    f"trained_segments={re_vae_bundle.get('n_segments')}"
)


## VAE bot generation (RE)

VAE only invents **64-event (dx, dy) segments**; assembly reuses `stitch_bot_game` + `dt_by_session` (same as stitch).

In [ ]:
from src.vae_bot import generate_vae_bot_game
from src.features import extract_features
from src.config import RNG_SEED

N_VAE_BOTS = len(games_df)
VAE_POOL_SEGMENTS = 256  # random VAE segments available to the stitcher per game

vae_rows = []
sample_vae_trajectories = []
vae_rng = np.random.default_rng(RNG_SEED + 3)

for i in range(N_VAE_BOTS):
    bot_mouse = generate_vae_bot_game(
        re_vae_bundle,
        dt_samples=re_dt_samples,
        dt_by_session=re_dt_by_session,
        target_duration_ms=re_target_ms,
        n_pool_segments=VAE_POOL_SEGMENTS,
        rng=vae_rng,
    )
    if len(sample_vae_trajectories) < N_PREVIEW:
        sample_vae_trajectories.append(bot_mouse.copy())

    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({
        "userId": -4,
        "gameId": f"vae_{i}",
        "source_file": f"synthetic_vae_{i}",
        "is_bot": 1,
        "bot_type": "vae",
    })
    vae_rows.append(feats)

bots_vae_df = pd.DataFrame(vae_rows)
print(f"Generated {len(bots_vae_df)} VAE bot games (pool={VAE_POOL_SEGMENTS} segs/game)")
print(bots_vae_df.head())


## VAE bot trajectory preview

In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_vae_trajectories):
    plot_trajectory(df, title=f"vae_{i}")
    print(f"\nvae_{i}, events={len(df)}")


## RE Human vs Bot classification

Includes **VAE** as a 4th bot tier (same features + RF). Cross-game diagnose for VAE can be wired next.


In [ ]:
from src.evaluation import train_bot_detector
from src.features import feature_cols

re_model_stitch, re_acc_stitch = train_bot_detector(
    games_df, bots_stitch_df, feature_cols, name="stitch"
)
re_model_smooth, re_acc_smooth = train_bot_detector(
    games_df, bots_smooth_df, feature_cols, name="smooth"
)
re_model_bezier, re_acc_bezier = train_bot_detector(
    games_df, bots_bezier_df, feature_cols, name="bezier"
)
re_model_vae, re_acc_vae = train_bot_detector(
    games_df, bots_vae_df, feature_cols, name="vae"
)

print("RE in-domain bot detection:")
print(f"  stitch: {re_acc_stitch:.2%}")
print(f"  smooth: {re_acc_smooth:.2%}")
print(f"  bezier: {re_acc_bezier:.2%}")
print(f"  vae:    {re_acc_vae:.2%}")



## Load LoL dataset

In [ ]:
from src.data import load_lol_match_windows
from src.config import LOL_MATCH_WINDOW_START_MIN, LOL_MATCH_WINDOW_END_MIN

# Fast-test knobs (None = full / one bot per human window)
RE_TRAIN_N = None
LOL_FILE_N = None
N_LOL_BOTS = None

# Quick peek: first match-aligned window (game clock 10–13 min)
_recs, _sum = load_lol_match_windows(max_keyloggers=1)
print("LoL match-window summary (smoke, 1 keylogger):", _sum)
if _recs:
    sample_lol = _recs[0]["mouse"]
    print(f"gameId={_recs[0]['gameId']}")
    print(
        f"Events: {len(sample_lol)}, duration: {sample_lol['time'].iloc[-1] / 60000:.2f} min "
        f"(target {LOL_MATCH_WINDOW_END_MIN - LOL_MATCH_WINDOW_START_MIN} min @ "
        f"{LOL_MATCH_WINDOW_START_MIN}-{LOL_MATCH_WINDOW_END_MIN})"
    )
    print(sample_lol.head())
else:
    sample_lol = None
    print("No window found for first keylogger (short match or sparse 10–13).")


## Load LoL sessions & extract features

In [ ]:
from src.features import extract_features
from src.data import load_lol_match_windows
from src.config import (
    LOL_MATCH_WINDOW_START_MIN,
    LOL_MATCH_WINDOW_END_MIN,
    LOL_MATCH_MIN_EVENTS,
)

# Optional cap for fast tests (None = all session keyloggers; set in LoL peek cell)
lol_records, lol_load_summary = load_lol_match_windows(
    window_start_min=LOL_MATCH_WINDOW_START_MIN,
    window_end_min=LOL_MATCH_WINDOW_END_MIN,
    min_events=LOL_MATCH_MIN_EVENTS,
    max_keyloggers=LOL_FILE_N,
)
print("LoL match-window load:", lol_load_summary)

lol_rows = []
lol_human_mice = []
lol_mouse_by_game = {}
for rec in lol_records:
    mouse = rec["mouse"]
    feats = extract_features(mouse)
    if feats is None:
        continue
    feats.update({
        "userId": rec["userId"],
        "gameId": rec["gameId"],
        "source_file": rec["source_file"],
        "session_date": rec["session_date"],
        "match_id": rec["match_id"],
        "participant": rec["participant"],
        "is_bot": 0,
        "bot_type": "human",
        "window_start_min": rec["window_start_min"],
        "window_end_min": rec["window_end_min"],
    })
    lol_rows.append(feats)
    lol_human_mice.append(mouse)
    lol_mouse_by_game[rec["gameId"]] = mouse

lol_games_df = pd.DataFrame(lol_rows)
print(
    f"Loaded {len(lol_games_df)} LoL human windows "
    f"(match-aligned {LOL_MATCH_WINDOW_START_MIN}–{LOL_MATCH_WINDOW_END_MIN} min; "
    f"was file-start first 3 min before)"
)
print(lol_games_df[["n_events", "total_movement", "avg_speed", "idle_ratio"]].describe())
print()
print(f"Red Eclipse — median n_events: {games_df['n_events'].median():.0f}")
print(f"LoL — median n_events: {lol_games_df['n_events'].median():.0f}")
print(f"LoL users: {lol_games_df['userId'].nunique()}, matches: {lol_games_df['match_id'].nunique()}")


## LoL trajectory preview

In [ ]:
from src.config import LOL_MATCH_WINDOW_START_MIN, LOL_MATCH_WINDOW_END_MIN
from src.plotting import plot_trajectory

# Preview first match-aligned window (same object as load)
if not lol_human_mice:
    raise RuntimeError("No LoL windows loaded — run the LoL feature cell first")

preview_lol = lol_human_mice[0].copy()
preview_lol["trajectory_x"] = preview_lol["dx"].cumsum()
preview_lol["trajectory_y"] = preview_lol["dy"].cumsum()
print(
    f"Preview: {lol_games_df.iloc[0]['gameId']} | "
    f"{len(preview_lol)} events | {preview_lol['time'].iloc[-1]/60000:.2f} min"
)
plot_trajectory(preview_lol, title=f"LoL match window {LOL_MATCH_WINDOW_START_MIN}-{LOL_MATCH_WINDOW_END_MIN} min")


## Cross-game transfer (RE train → LoL test)

In [ ]:
from src.features import cross_game_feature_cols
from src.evaluation import train_bot_detector

re_human = games_df.head(RE_TRAIN_N) if RE_TRAIN_N else games_df
re_stitch = bots_stitch_df.head(RE_TRAIN_N) if RE_TRAIN_N else bots_stitch_df
re_smooth = bots_smooth_df.head(RE_TRAIN_N) if RE_TRAIN_N else bots_smooth_df
re_bezier = bots_bezier_df.head(RE_TRAIN_N) if RE_TRAIN_N else bots_bezier_df
re_vae = bots_vae_df.head(RE_TRAIN_N) if RE_TRAIN_N else bots_vae_df

print("=== RE model trained on STITCH bots ===")
re_model_stitch, re_acc_stitch = train_bot_detector(re_human, re_stitch, cross_game_feature_cols, name="stitch")
print()
print("=== RE model trained on SMOOTH bots ===")
re_model_smooth, re_acc_smooth = train_bot_detector(re_human, re_smooth, cross_game_feature_cols, name="smooth")
print()
print("=== RE model trained on BEZIER bots ===")
re_model_bezier, re_acc_bezier = train_bot_detector(re_human, re_bezier, cross_game_feature_cols, name="bezier")

print()
print("=== RE model trained on VAE bots ===")
re_model_vae, re_acc_vae = train_bot_detector(re_human, re_vae, cross_game_feature_cols, name="vae")


## LoL data human identifier

In [ ]:
# test will LoL humans identified as bot
for name, model in [
    ("stitch", re_model_stitch),
    ("smooth", re_model_smooth),
    ("bezier", re_model_bezier),
    ("vae", re_model_vae),
]:
    pred = model.predict(lol_games_df[cross_game_feature_cols])
    print(f"[{name} model] LoL human false-positive rate: {pred.mean():.2%} ({pred.sum()}/{len(pred)} identified as bot)")



## LoL stitch bot generation

In [ ]:
from src.bots import (
    build_segments,
    stitch_bot_game,
    collect_human_motion_samples,
    median_trace_duration_ms,
)
from src.features import extract_features

# Reuse match-aligned windows from the LoL feature cell
lol_segment_pool = []
if not lol_human_mice:
    raise RuntimeError("lol_human_mice empty — run LoL match-window feature cell first")

lol_bot_rng = np.random.default_rng(RNG_SEED + 1)
for mouse in lol_human_mice:
    lol_segment_pool.extend(build_segments(mouse, rng=lol_bot_rng))

lol_motion = collect_human_motion_samples(lol_human_mice, rng=lol_bot_rng)
lol_dt_samples = lol_motion["dt_samples"]
lol_dt_by_session = lol_motion["dt_by_session"]
lol_target_ms = median_trace_duration_ms(lol_human_mice)
print(
    f"LoL dt n={len(lol_dt_samples)} sessions={len(lol_dt_by_session)} median={np.median(lol_dt_samples):.2f}ms | "
    f"step median={np.median(lol_motion['step_samples']):.3f} | "
    f"target={lol_target_ms/1000:.1f}s"
)

n_lol_bots = N_LOL_BOTS if N_LOL_BOTS else len(lol_games_df)
lol_stitch_rows = []
sample_lol_stitch_trajectories = []

for i in range(n_lol_bots):
    bot_mouse = stitch_bot_game(
        lol_segment_pool,
        dt_samples=lol_dt_samples,
        dt_by_session=lol_dt_by_session,
        target_duration_ms=lol_target_ms,
        rng=lol_bot_rng,
    )
    if len(sample_lol_stitch_trajectories) < N_PREVIEW:
        sample_lol_stitch_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({"userId": -1, "gameId": f"lol_stitch_{i}", "is_bot": 1, "bot_type": "stitch"})
    lol_stitch_rows.append(feats)

lol_bots_stitch_df = pd.DataFrame(lol_stitch_rows)
print(f"LoL stitch bots: {len(lol_bots_stitch_df)} (target {lol_target_ms/1000:.1f}s each)")



## LoL stitch bot trajectory preview

In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_lol_stitch_trajectories):
    print(f"lol_stitch_{i}, events={len(df)}")
    plot_trajectory(df, title=f"lol_stitch_{i}")

## LoL smooth bot generation

In [ ]:
from src.bots import (
    estimate_smooth_params,
    generate_smooth_bot_game,
    smooth_params_for_print,
)

lol_median_events = int(lol_games_df["n_events"].median())
n_lol_smooth = N_LOL_BOTS if N_LOL_BOTS else len(lol_games_df)

lol_smooth_params = estimate_smooth_params(lol_games_df, **lol_motion)
print(f"LoL smooth params: {smooth_params_for_print(lol_smooth_params)}")
print(f"(RE smooth params for comparison: {smooth_params_for_print(re_smooth_params)})")

lol_smooth_rows = []
sample_lol_smooth_trajectories = []
for i in range(n_lol_smooth):
    bot_mouse = generate_smooth_bot_game(
        n_events=lol_median_events, seed=RNG_SEED + 100 + i, **lol_smooth_params
    )
    if len(sample_lol_smooth_trajectories) < N_PREVIEW:
        sample_lol_smooth_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({"userId": -2, "gameId": f"lol_smooth_{i}", "is_bot": 1, "bot_type": "smooth"})
    lol_smooth_rows.append(feats)

lol_bots_smooth_df = pd.DataFrame(lol_smooth_rows)
print(f"LoL smooth bots: {len(lol_bots_smooth_df)} (n_events={lol_median_events})")


## LoL smooth bot trajectory preview

In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_lol_smooth_trajectories):
    plot_trajectory(df, title=f"lol_smooth_{i}")
    print(f"lol_smooth_{i}, events={len(df)}")


## LoL Bézier bot generation


In [ ]:
from src.bots import (
    estimate_bezier_params,
    generate_bezier_bot_game,
    bezier_params_for_print,
)

lol_median_events = int(lol_games_df["n_events"].median())
n_lol_bezier = N_LOL_BOTS if N_LOL_BOTS else len(lol_games_df)

lol_bezier_params = estimate_bezier_params(lol_games_df, **lol_motion)
print(f"LoL bezier params: {bezier_params_for_print(lol_bezier_params)}")
print(f"(RE bezier params for comparison: {bezier_params_for_print(re_bezier_params)})")

lol_bezier_rows = []
sample_lol_bezier_trajectories = []
for i in range(n_lol_bezier):
    bot_mouse = generate_bezier_bot_game(
        n_events=lol_median_events, seed=RNG_SEED + 150 + i, **lol_bezier_params
    )
    if len(sample_lol_bezier_trajectories) < N_PREVIEW:
        sample_lol_bezier_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({"userId": -3, "gameId": f"lol_bezier_{i}", "is_bot": 1, "bot_type": "bezier"})
    lol_bezier_rows.append(feats)

lol_bots_bezier_df = pd.DataFrame(lol_bezier_rows)
print(f"LoL bezier bots: {len(lol_bots_bezier_df)} (n_events={lol_median_events})")



## LoL Bézier bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_lol_bezier_trajectories):
    plot_trajectory(df, title=f"lol_bezier_{i}")
    print(f"lol_bezier_{i}, events={len(df)}")



## LoL VAE bot — train once / load weights

Default: load `artifacts/vae_lol_v3.pt`. Set `LOL_VAE_FORCE_RETRAIN = True` only if protocol/data changed.


In [ ]:
from pathlib import Path
import numpy as np

from src.config import RNG_SEED
from src.vae_bot import ensure_vae_bundle, DEFAULT_LOL_WEIGHTS

LOL_VAE_FORCE_RETRAIN = False
LOL_VAE_WEIGHTS_PATH = DEFAULT_LOL_WEIGHTS

lol_step_median = float(np.median(lol_motion["step_samples"]))
lol_vae_bundle = ensure_vae_bundle(
    lol_human_mice,
    lol_step_median,
    path=LOL_VAE_WEIGHTS_PATH,
    force_retrain=LOL_VAE_FORCE_RETRAIN,
    seed=RNG_SEED + 1,
)
print(
    f"LoL VAE ready | path={Path(LOL_VAE_WEIGHTS_PATH)} | "
    f"seg_len={lol_vae_bundle['seg_len']} z={lol_vae_bundle['z_dim']} "
    f"step_median={lol_vae_bundle['step_median']:.4f} "
    f"trained_segments={lol_vae_bundle.get('n_segments')}"
)


## LoL VAE bot generation

VAE segments → same `stitch_bot_game` assembly + `dt_by_session` as LoL stitch.


In [ ]:
from src.vae_bot import generate_vae_bot_game
from src.features import extract_features
from src.config import RNG_SEED

n_lol_vae = N_LOL_BOTS if N_LOL_BOTS else len(lol_games_df)
LOL_VAE_POOL_SEGMENTS = 256

lol_vae_rows = []
sample_lol_vae_trajectories = []
lol_vae_rng = np.random.default_rng(RNG_SEED + 4)

for i in range(n_lol_vae):
    bot_mouse = generate_vae_bot_game(
        lol_vae_bundle,
        dt_samples=lol_dt_samples,
        dt_by_session=lol_dt_by_session,
        target_duration_ms=lol_target_ms,
        n_pool_segments=LOL_VAE_POOL_SEGMENTS,
        rng=lol_vae_rng,
    )
    if len(sample_lol_vae_trajectories) < N_PREVIEW:
        sample_lol_vae_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({
        "userId": -4,
        "gameId": f"lol_vae_{i}",
        "is_bot": 1,
        "bot_type": "vae",
    })
    lol_vae_rows.append(feats)

lol_bots_vae_df = pd.DataFrame(lol_vae_rows)
print(f"LoL VAE bots: {len(lol_bots_vae_df)} (pool={LOL_VAE_POOL_SEGMENTS} segs/game)")
print(lol_bots_vae_df.head())


## LoL VAE bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_lol_vae_trajectories):
    plot_trajectory(df, title=f"lol_vae_{i}")
    print(f"\nlol_vae_{i}, events={len(df)}")


## LoL in-domain sanity check

In [ ]:
from src.evaluation import train_bot_detector
from src.features import feature_cols

_, lol_re_acc_stitch = train_bot_detector(
    lol_games_df, lol_bots_stitch_df, feature_cols, name="LoL stitch"
)
print()
_, lol_re_acc_smooth = train_bot_detector(
    lol_games_df, lol_bots_smooth_df, feature_cols, name="LoL smooth"
)
print()
_, lol_re_acc_bezier = train_bot_detector(
    lol_games_df, lol_bots_bezier_df, feature_cols, name="LoL bezier"
)
print()
_, lol_re_acc_vae = train_bot_detector(
    lol_games_df, lol_bots_vae_df, feature_cols, name="LoL vae"
)

print()
print("LoL in-domain bot detection (train+test on LoL):")
print(f"  stitch: {lol_re_acc_stitch:.2%}")
print(f"  smooth: {lol_re_acc_smooth:.2%}")
print(f"  bezier: {lol_re_acc_bezier:.2%}")
print(f"  vae:    {lol_re_acc_vae:.2%}")



## Feature scale comparison (RE vs LoL)

In [ ]:
from src.features import cross_game_feature_cols

cols = cross_game_feature_cols

print("Feature medians (RE human vs LoL human vs LoL bots):")
compare = pd.DataFrame({
    "RE_human": games_df[cols].median(),
    "LoL_human": lol_games_df[cols].median(),
    "LoL_stitch": lol_bots_stitch_df[cols].median(),
    "LoL_smooth": lol_bots_smooth_df[cols].median(),
    "LoL_bezier": lol_bots_bezier_df[cols].median(),
    "LoL_vae": lol_bots_vae_df[cols].median(),
}).round(3)
print(compare)



## Zero-shot diagnose (raw features, RE → LoL)

In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import cross_game_feature_cols

print("=== Raw features: true zero-shot RE -> LoL ===")
print("(threshold from LoL humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", re_model_stitch, lol_games_df, lol_bots_stitch_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "smooth", re_model_smooth, lol_games_df, lol_bots_smooth_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "bezier", re_model_bezier, lol_games_df, lol_bots_bezier_df,
    cross_game_feature_cols, title_suffix="raw",
)

print()
_ = diagnose_cross_game(
    "vae", re_model_vae, lol_games_df, lol_bots_vae_df,
    cross_game_feature_cols, title_suffix="raw",
)


## Scale-invariant train (RE)

Train RF on unitless ratio features (same RE subset as raw cross-game models).



In [ ]:
from src.evaluation import train_bot_detector
from src.features import to_scale_invariant, SCALE_INVARIANT_COLS
from src.config import RNG_SEED

# Same RE training subset as raw cross-game models when RE_TRAIN_N is set
re_si_human = to_scale_invariant(re_human)
re_si_stitch = to_scale_invariant(re_stitch)
re_si_smooth = to_scale_invariant(re_smooth)
re_si_bezier = to_scale_invariant(re_bezier)
re_si_vae = to_scale_invariant(re_vae)

print("=== Train on Red Eclipse (scale-invariant features) ===")
m_si_stitch, _ = train_bot_detector(
    re_si_human, re_si_stitch, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="scale-invariant stitch",
    show_feature_importance=False,
)
print()
m_si_smooth, _ = train_bot_detector(
    re_si_human, re_si_smooth, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="scale-invariant smooth",
    show_feature_importance=False,
)
print()
m_si_bezier, _ = train_bot_detector(
    re_si_human, re_si_bezier, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="scale-invariant bezier",
    show_feature_importance=False,
)

print()
m_si_vae, _ = train_bot_detector(
    re_si_human, re_si_vae, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="scale-invariant vae",
    show_feature_importance=False,
)


## Zero-shot diagnose (scale-invariant features, RE → LoL)

Same `diagnose_zero_shot` suite as raw: AUC, P(bot), @0.5, human-calibrated threshold, sweep.



In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import to_scale_invariant, SCALE_INVARIANT_COLS

lol_si_human = to_scale_invariant(lol_games_df)
lol_si_stitch = to_scale_invariant(lol_bots_stitch_df)
lol_si_smooth = to_scale_invariant(lol_bots_smooth_df)
lol_si_bezier = to_scale_invariant(lol_bots_bezier_df)
lol_si_vae = to_scale_invariant(lol_bots_vae_df)

print("=== Scale-invariant: true zero-shot RE -> LoL ===")
print("(threshold from LoL humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", m_si_stitch, lol_si_human, lol_si_stitch,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)
print()
_ = diagnose_cross_game(
    "smooth", m_si_smooth, lol_si_human, lol_si_smooth,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)
print()
_ = diagnose_cross_game(
    "bezier", m_si_bezier, lol_si_human, lol_si_bezier,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)

print()
_ = diagnose_cross_game(
    "vae", m_si_vae, lol_si_human, lol_si_vae,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)


## RE→LoL Bézier inversion diagnostic

Why raw AUC≈0.1 (scores flipped): compare feature medians RE vs LoL, and which high-importance features reverse human↔bot order across games.


In [ ]:
import pandas as pd
from src.features import cross_game_feature_cols, to_scale_invariant, SCALE_INVARIANT_COLS

raw_cols = cross_game_feature_cols

# --- (1) Feature medians: RE human / RE bezier / LoL human / LoL bezier ---
print("=== Raw 7-feature medians ===")
raw_med = pd.DataFrame({
    "RE_human": games_df[raw_cols].median(),
    "RE_bezier": bots_bezier_df[raw_cols].median(),
    "LoL_human": lol_games_df[raw_cols].median(),
    "LoL_bezier": lol_bots_bezier_df[raw_cols].median(),
}).round(4)
# signed gap human - bot (same game); flip if RE and LoL gaps have opposite sign
raw_med["gap_RE"] = (raw_med["RE_human"] - raw_med["RE_bezier"]).round(4)
raw_med["gap_LoL"] = (raw_med["LoL_human"] - raw_med["LoL_bezier"]).round(4)
raw_med["sign_flip"] = (raw_med["gap_RE"] * raw_med["gap_LoL"]) < 0
print(raw_med)
print()

re_sf = to_scale_invariant(games_df)
re_bz_sf = to_scale_invariant(bots_bezier_df)
lol_sf = to_scale_invariant(lol_games_df)
lol_bz_sf = to_scale_invariant(lol_bots_bezier_df)

print("=== Scale-invariant 4-feature medians ===")
sf_med = pd.DataFrame({
    "RE_human": re_sf[SCALE_INVARIANT_COLS].median(),
    "RE_bezier": re_bz_sf[SCALE_INVARIANT_COLS].median(),
    "LoL_human": lol_sf[SCALE_INVARIANT_COLS].median(),
    "LoL_bezier": lol_bz_sf[SCALE_INVARIANT_COLS].median(),
}).round(4)
sf_med["gap_RE"] = (sf_med["RE_human"] - sf_med["RE_bezier"]).round(4)
sf_med["gap_LoL"] = (sf_med["LoL_human"] - sf_med["LoL_bezier"]).round(4)
sf_med["sign_flip"] = (sf_med["gap_RE"] * sf_med["gap_LoL"]) < 0
print(sf_med)
print()

# --- (2) Feature importance of the RE-trained bezier detectors ---
print("=== RE bezier model importance (raw 7-feat, used in RE→LoL raw) ===")
imp_raw = pd.Series(
    re_model_bezier.feature_importances_, index=raw_cols
).sort_values(ascending=False)
print(imp_raw.round(4))
print()

print("=== RE bezier model importance (scale-invariant, used in RE→LoL SF) ===")
imp_sf = pd.Series(
    m_si_bezier.feature_importances_, index=SCALE_INVARIANT_COLS
).sort_values(ascending=False)
print(imp_sf.round(4))
print()

# --- Cross-read: high importance ∩ sign flip ---
print("=== Suspects: importance rank + sign_flip ===")
print("Raw:")
for feat, imp in imp_raw.items():
    flip = bool(raw_med.loc[feat, "sign_flip"])
    print(f"  {feat:16s}  imp={imp:.3f}  flip={flip}  "
          f"gap_RE={raw_med.loc[feat, 'gap_RE']:+.4f}  gap_LoL={raw_med.loc[feat, 'gap_LoL']:+.4f}")
print("Scale-invariant:")
for feat, imp in imp_sf.items():
    flip = bool(sf_med.loc[feat, "sign_flip"])
    print(f"  {feat:16s}  imp={imp:.3f}  flip={flip}  "
          f"gap_RE={sf_med.loc[feat, 'gap_RE']:+.4f}  gap_LoL={sf_med.loc[feat, 'gap_LoL']:+.4f}")




## Feature importance (scale-invariant models)



In [ ]:
import pandas as pd
from src.features import SCALE_INVARIANT_COLS

imp = pd.Series(m_si_stitch.feature_importances_, index=SCALE_INVARIANT_COLS).sort_values(ascending=False)
print("=== Feature importance (scale-invariant stitch model) ===")
print(imp)
print()
imp2 = pd.Series(m_si_smooth.feature_importances_, index=SCALE_INVARIANT_COLS).sort_values(ascending=False)
print("=== Feature importance (scale-invariant smooth model) ===")
print(imp2)
print()
imp3 = pd.Series(m_si_bezier.feature_importances_, index=SCALE_INVARIANT_COLS).sort_values(ascending=False)
print("=== Feature importance (scale-invariant bezier model) ===")
print(imp3)

print()
imp4 = pd.Series(m_si_vae.feature_importances_, index=SCALE_INVARIANT_COLS).sort_values(ascending=False)
print("=== Feature importance (scale-invariant vae model) ===")
print(imp4)


## CSGO load: eye_vector → (dx, dy, time)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from src.config import CSGO_DATA_ROOT
from src.data_csgo import (
    check_axis_convention,
    eye_vectors_to_mouse_df,
    validate_real_roundtrip,
)

USECOLS = ["time", "eyeVectorX", "eyeVectorY", "eyeVectorZ"]

def find_gameflt_files(root=CSGO_DATA_ROOT):
    files = sorted(Path(root).rglob("gameFlt.csv"))
    print(f"Found {len(files)} gameFlt.csv under {root}")
    return files

def load_eye_csv(path):
    df = pd.read_csv(path, usecols=USECOLS)
    finite = np.isfinite(df[["eyeVectorX", "eyeVectorY", "eyeVectorZ"]]).all(axis=1)
    return df.loc[finite].reset_index(drop=True)

def convert_one(path):
    flt = load_eye_csv(path)
    mouse_df, meta = eye_vectors_to_mouse_df(
        flt["time"], flt["eyeVectorX"], flt["eyeVectorY"], flt["eyeVectorZ"]
    )
    return mouse_df, meta, flt

gameflt_paths = find_gameflt_files()
assert gameflt_paths, f"No gameFlt.csv under {CSGO_DATA_ROOT}"

first_path = gameflt_paths[0]
print(f"\n=== First-file checks: {first_path} ===")

mouse0, meta0, flt0 = convert_one(first_path)
print("convert meta:", meta0)
print(mouse0.head())

axis_ok = check_axis_convention(flt0["eyeVectorY"])
rt = validate_real_roundtrip(
    flt0["eyeVectorX"], flt0["eyeVectorY"], flt0["eyeVectorZ"]
)

if not (axis_ok and rt["ok"]):
    raise RuntimeError(
        "First-file checks failed — stop before converting all files. "
        f"axis_ok={axis_ok}, roundtrip_ok={rt['ok']}"
    )

print("\nFirst file PASSED. Converting all gameFlt.csv ...")

csgo_mouse = {}
rows = []
for i, path in enumerate(gameflt_paths, 1):
    # path like .../S001/P3/gameFlt.csv
    participant = path.parent.name          # P3
    session = path.parent.parent.name       # S001
    key = (session, participant)
    try:
        mouse_df, meta, _ = convert_one(path)
    except Exception as e:
        print(f"  SKIP {key}: {e}")
        continue
    csgo_mouse[key] = mouse_df
    rows.append({
        "session": session,
        "participant": participant,
        "n_out": meta["n_out"],
        "teleport_frac": meta["teleport_frac"],
        "path": str(path),
    })
    if i % 50 == 0 or i == len(gameflt_paths):
        print(f"  converted {i}/{len(gameflt_paths)}")

csgo_convert_summary = pd.DataFrame(rows)
print(f"\nDone: {len(csgo_mouse)} traces")
print(csgo_convert_summary.head())
print(
    "n_out median:", csgo_convert_summary["n_out"].median(),
    "| teleport_frac median:", f"{csgo_convert_summary['teleport_frac'].median():.2%}",
)

## CSGO sessions & extract features


In [ ]:
from pathlib import Path

from src.features import extract_features
from src.config import CSGO_DATA_ROOT, CSGO_WINDOW_MIN
from src.data_csgo import window_mouse_round_alive

print(
    f"Extracting features from {len(csgo_mouse)} CSGO traces "
    f"(Round2+alive, {CSGO_WINDOW_MIN} min)"
)

csgo_mouse_win = {}
csgo_rows = []
csgo_window_meta = []
n_skip = 0

for (session, participant), mouse in csgo_mouse.items():
    session_dir = Path(CSGO_DATA_ROOT) / session / participant
    win, meta = window_mouse_round_alive(
        mouse, session_dir, window_min=CSGO_WINDOW_MIN
    )
    csgo_window_meta.append({"session": session, "participant": participant, **meta})
    if not meta.get("ok") or win is None:
        n_skip += 1
        continue

    feats = extract_features(win)
    if feats is None:
        n_skip += 1
        continue

    csgo_mouse_win[(session, participant)] = win
    feats.update({
        "userId": f"csgo_{session}_{participant}",
        "gameId": f"{session}_{participant}",
        "session": session,
        "participant": participant,
        "is_bot": 0,
        "bot_type": "human",
        "round_n": meta["round_n"],
        "alive_frac_in_window": meta["alive_frac_in_window"],
    })
    csgo_rows.append(feats)

csgo_games_df = pd.DataFrame(csgo_rows)
csgo_window_meta_df = pd.DataFrame(csgo_window_meta)

print(
    f"Loaded {len(csgo_games_df)} CSGO human sessions "
    f"(Round2+alive, skipped {n_skip})"
)
print(
    "alive_frac_in_window median:",
    f"{csgo_games_df['alive_frac_in_window'].median():.1%}",
)
print(csgo_games_df[["n_events", "total_movement", "avg_speed", "idle_ratio"]].describe())
print()
print(f"Red Eclipse — median n_events: {games_df['n_events'].median():.0f}")
print(f"LoL — median n_events: {lol_games_df['n_events'].median():.0f}")
print(f"CSGO — median n_events: {csgo_games_df['n_events'].median():.0f}")
print(csgo_games_df.head())


## CSGO trajectory preview


In [ ]:
from src.plotting import plot_trajectory
from src.config import CSGO_WINDOW_MIN

preview_keys = list(csgo_mouse_win.keys())[:5]

for session, participant in preview_keys:
    mouse = csgo_mouse_win[(session, participant)]
    plot_trajectory(
        mouse,
        title=f"CSGO {session}/{participant} (Round2+alive, {CSGO_WINDOW_MIN} min)",
    )
    print(f"\n{session}/{participant}, events={len(mouse)}")


## CSGO stitch bot generation


In [ ]:
from src.bots import (
    build_segments,
    stitch_bot_game,
    collect_human_motion_samples,
    median_trace_duration_ms,
)
from src.features import extract_features
from src.config import RNG_SEED

N_PREVIEW = 5
N_CSGO_BOTS = None  # None = one bot per CSGO human session

csgo_bot_rng = np.random.default_rng(RNG_SEED + 2)
csgo_segment_pool = []
for mouse in csgo_mouse_win.values():
    csgo_segment_pool.extend(build_segments(mouse, rng=csgo_bot_rng))

csgo_motion = collect_human_motion_samples(csgo_mouse_win.values(), rng=csgo_bot_rng)
csgo_dt_samples = csgo_motion["dt_samples"]
csgo_dt_by_session = csgo_motion["dt_by_session"]
csgo_target_ms = median_trace_duration_ms(csgo_mouse_win.values())
print(
    f"CSGO segment pool: {len(csgo_segment_pool)} segments "
    f"from {len(csgo_mouse_win)} Round2+alive traces | "
    f"dt n={len(csgo_dt_samples)} sessions={len(csgo_dt_by_session)} median={np.median(csgo_dt_samples):.2f}ms | "
    f"step median={np.median(csgo_motion['step_samples']):.3f} | "
    f"target={csgo_target_ms/1000:.1f}s"
)

n_csgo_bots = N_CSGO_BOTS if N_CSGO_BOTS else len(csgo_games_df)
csgo_stitch_rows = []
sample_csgo_stitch_trajectories = []

for i in range(n_csgo_bots):
    bot_mouse = stitch_bot_game(
        csgo_segment_pool,
        dt_samples=csgo_dt_samples,
        dt_by_session=csgo_dt_by_session,
        target_duration_ms=csgo_target_ms,
        rng=csgo_bot_rng,
    )
    if len(sample_csgo_stitch_trajectories) < N_PREVIEW:
        sample_csgo_stitch_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({
        "userId": -1,
        "gameId": f"csgo_stitch_{i}",
        "is_bot": 1,
        "bot_type": "stitch",
    })
    csgo_stitch_rows.append(feats)

csgo_bots_stitch_df = pd.DataFrame(csgo_stitch_rows)
print(f"CSGO stitch bots: {len(csgo_bots_stitch_df)} (target {csgo_target_ms/1000:.1f}s each)")
print(csgo_bots_stitch_df.head())



## CSGO stitch bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_csgo_stitch_trajectories):
    print(f"csgo_stitch_{i}, events={len(df)}")
    plot_trajectory(df, title=f"csgo_stitch_{i}")


## CSGO smooth bot generation


In [ ]:
from src.bots import (
    estimate_smooth_params,
    generate_smooth_bot_game,
    smooth_params_for_print,
)
from src.features import extract_features
from src.config import RNG_SEED

csgo_median_events = int(csgo_games_df["n_events"].median())
n_csgo_smooth = N_CSGO_BOTS if N_CSGO_BOTS else len(csgo_games_df)

csgo_smooth_params = estimate_smooth_params(csgo_games_df, **csgo_motion)
print(f"CSGO smooth params: {smooth_params_for_print(csgo_smooth_params)}")

csgo_smooth_rows = []
sample_csgo_smooth_trajectories = []
for i in range(n_csgo_smooth):
    # round_deltas=False: CSGO dx/dy are degrees (often << 1); integer round would wipe them
    bot_mouse = generate_smooth_bot_game(
        n_events=csgo_median_events,
        seed=RNG_SEED + 200 + i,
        round_deltas=False,
        **csgo_smooth_params,
    )
    if len(sample_csgo_smooth_trajectories) < N_PREVIEW:
        sample_csgo_smooth_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({
        "userId": -2,
        "gameId": f"csgo_smooth_{i}",
        "is_bot": 1,
        "bot_type": "smooth",
    })
    csgo_smooth_rows.append(feats)

csgo_bots_smooth_df = pd.DataFrame(csgo_smooth_rows)
print(f"CSGO smooth bots: {len(csgo_bots_smooth_df)} (n_events={csgo_median_events})")
print(csgo_bots_smooth_df.head())


## CSGO smooth bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_csgo_smooth_trajectories):
    plot_trajectory(df, title=f"csgo_smooth_{i}")
    print(f"csgo_smooth_{i}, events={len(df)}")


## CSGO Bézier bot generation


In [ ]:
from src.bots import (
    estimate_bezier_params,
    generate_bezier_bot_game,
    bezier_params_for_print,
)
from src.features import extract_features
from src.config import RNG_SEED

csgo_median_events = int(csgo_games_df["n_events"].median())
n_csgo_bezier = N_CSGO_BOTS if N_CSGO_BOTS else len(csgo_games_df)

csgo_bezier_params = estimate_bezier_params(csgo_games_df, **csgo_motion)
print(f"CSGO bezier params: {bezier_params_for_print(csgo_bezier_params)}")

csgo_bezier_rows = []
sample_csgo_bezier_trajectories = []
for i in range(n_csgo_bezier):
    # round_deltas=False: CSGO dx/dy are degrees (often << 1); integer round would wipe them
    bot_mouse = generate_bezier_bot_game(
        n_events=csgo_median_events,
        seed=RNG_SEED + 250 + i,
        round_deltas=False,
        **csgo_bezier_params,
    )
    if len(sample_csgo_bezier_trajectories) < N_PREVIEW:
        sample_csgo_bezier_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({
        "userId": -3,
        "gameId": f"csgo_bezier_{i}",
        "is_bot": 1,
        "bot_type": "bezier",
    })
    csgo_bezier_rows.append(feats)

csgo_bots_bezier_df = pd.DataFrame(csgo_bezier_rows)
print(f"CSGO bezier bots: {len(csgo_bots_bezier_df)} (n_events={csgo_median_events})")
print(csgo_bots_bezier_df.head())



## CSGO Bézier bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_csgo_bezier_trajectories):
    plot_trajectory(df, title=f"csgo_bezier_{i}")
    print(f"csgo_bezier_{i}, events={len(df)}")



## CSGO VAE bot — train once / load weights

Default: load `artifacts/vae_csgo_v1.pt`. Set `CSGO_VAE_FORCE_RETRAIN = True` only if protocol/data changed.


In [ ]:
from pathlib import Path
import numpy as np

from src.config import RNG_SEED
from src.vae_bot import ensure_vae_bundle, DEFAULT_CSGO_WEIGHTS

CSGO_VAE_FORCE_RETRAIN = False
CSGO_VAE_WEIGHTS_PATH = DEFAULT_CSGO_WEIGHTS

csgo_step_median = float(np.median(csgo_motion["step_samples"]))
csgo_vae_bundle = ensure_vae_bundle(
    list(csgo_mouse_win.values()),
    csgo_step_median,
    path=CSGO_VAE_WEIGHTS_PATH,
    force_retrain=CSGO_VAE_FORCE_RETRAIN,
    seed=RNG_SEED + 2,
)
print(
    f"CSGO VAE ready | path={Path(CSGO_VAE_WEIGHTS_PATH)} | "
    f"seg_len={csgo_vae_bundle['seg_len']} z={csgo_vae_bundle['z_dim']} "
    f"step_median={csgo_vae_bundle['step_median']:.4f} "
    f"trained_segments={csgo_vae_bundle.get('n_segments')}"
)


## CSGO VAE bot generation

VAE segments → `stitch_bot_game` + CSGO `dt_by_session` (degrees scale via that game's `step_median`).


In [ ]:
from src.vae_bot import generate_vae_bot_game
from src.features import extract_features
from src.config import RNG_SEED

n_csgo_vae = N_CSGO_BOTS if N_CSGO_BOTS else len(csgo_games_df)
CSGO_VAE_POOL_SEGMENTS = 256

csgo_vae_rows = []
sample_csgo_vae_trajectories = []
csgo_vae_rng = np.random.default_rng(RNG_SEED + 5)

for i in range(n_csgo_vae):
    bot_mouse = generate_vae_bot_game(
        csgo_vae_bundle,
        dt_samples=csgo_dt_samples,
        dt_by_session=csgo_dt_by_session,
        target_duration_ms=csgo_target_ms,
        n_pool_segments=CSGO_VAE_POOL_SEGMENTS,
        rng=csgo_vae_rng,
    )
    if len(sample_csgo_vae_trajectories) < N_PREVIEW:
        sample_csgo_vae_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({
        "userId": -4,
        "gameId": f"csgo_vae_{i}",
        "is_bot": 1,
        "bot_type": "vae",
    })
    csgo_vae_rows.append(feats)

csgo_bots_vae_df = pd.DataFrame(csgo_vae_rows)
print(f"CSGO VAE bots: {len(csgo_bots_vae_df)} (pool={CSGO_VAE_POOL_SEGMENTS} segs/game)")
print(csgo_bots_vae_df.head())


## CSGO VAE bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_csgo_vae_trajectories):
    plot_trajectory(df, title=f"csgo_vae_{i}")
    print(f"\ncsgo_vae_{i}, events={len(df)}")


## CSGO in-domain sanity check

Uses Round2+alive human windows and bots built from the same cut.


In [ ]:
from src.evaluation import train_bot_detector
from src.features import feature_cols

csgo_model_stitch, csgo_acc_stitch = train_bot_detector(
    csgo_games_df, csgo_bots_stitch_df, feature_cols, name="CSGO stitch"
)
print()
csgo_model_smooth, csgo_acc_smooth = train_bot_detector(
    csgo_games_df, csgo_bots_smooth_df, feature_cols, name="CSGO smooth"
)
print()
csgo_model_bezier, csgo_acc_bezier = train_bot_detector(
    csgo_games_df, csgo_bots_bezier_df, feature_cols, name="CSGO bezier"
)
print()
csgo_model_vae, csgo_acc_vae = train_bot_detector(
    csgo_games_df, csgo_bots_vae_df, feature_cols, name="CSGO vae"
)

print()
print("CSGO in-domain bot detection (train+test on CSGO):")
print(f"  stitch: {csgo_acc_stitch:.2%}")
print(f"  smooth: {csgo_acc_smooth:.2%}")
print(f"  bezier: {csgo_acc_bezier:.2%}")
print(f"  vae:    {csgo_acc_vae:.2%}")



## Zero-shot diagnose (raw features, RE → CSGO)

In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import cross_game_feature_cols

print("=== Raw features: true zero-shot RE -> CSGO ===")
print("(threshold from CSGO humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", re_model_stitch, csgo_games_df, csgo_bots_stitch_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "smooth", re_model_smooth, csgo_games_df, csgo_bots_smooth_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "bezier", re_model_bezier, csgo_games_df, csgo_bots_bezier_df,
    cross_game_feature_cols, title_suffix="raw",
)

print()
_ = diagnose_cross_game(
    "vae", re_model_vae, csgo_games_df, csgo_bots_vae_df,
    cross_game_feature_cols, title_suffix="raw",
)


## Scale-invariant train (RE) — for CSGO transfer


In [ ]:
from src.evaluation import train_bot_detector
from src.features import to_scale_invariant, SCALE_INVARIANT_COLS
from src.config import RNG_SEED

re_si_human = to_scale_invariant(re_human)
re_si_stitch = to_scale_invariant(re_stitch)
re_si_smooth = to_scale_invariant(re_smooth)
re_si_bezier = to_scale_invariant(re_bezier)
re_si_vae = to_scale_invariant(re_vae)

print("=== Train on Red Eclipse (scale-invariant features) ===")
m_si_stitch, _ = train_bot_detector(
    re_si_human, re_si_stitch, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="scale-invariant stitch",
    show_feature_importance=False,
)
print()
m_si_smooth, _ = train_bot_detector(
    re_si_human, re_si_smooth, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="scale-invariant smooth",
    show_feature_importance=False,
)
print()
m_si_bezier, _ = train_bot_detector(
    re_si_human, re_si_bezier, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="scale-invariant bezier",
    show_feature_importance=False,
)

print()
m_si_vae, _ = train_bot_detector(
    re_si_human, re_si_vae, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="scale-invariant vae",
    show_feature_importance=False,
)


## Zero-shot diagnose (scale-invariant features, RE → CSGO)


In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import to_scale_invariant, SCALE_INVARIANT_COLS

csgo_si_human = to_scale_invariant(csgo_games_df)
csgo_si_stitch = to_scale_invariant(csgo_bots_stitch_df)
csgo_si_smooth = to_scale_invariant(csgo_bots_smooth_df)
csgo_si_bezier = to_scale_invariant(csgo_bots_bezier_df)
csgo_si_vae = to_scale_invariant(csgo_bots_vae_df)

print("=== Scale-invariant: true zero-shot RE -> CSGO ===")
print("(threshold from CSGO humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", m_si_stitch, csgo_si_human, csgo_si_stitch,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)
print()
_ = diagnose_cross_game(
    "smooth", m_si_smooth, csgo_si_human, csgo_si_smooth,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)
print()
_ = diagnose_cross_game(
    "bezier", m_si_bezier, csgo_si_human, csgo_si_bezier,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)

print()
_ = diagnose_cross_game(
    "vae", m_si_vae, csgo_si_human, csgo_si_vae,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)


## Feature scale comparison (RE vs CSGO)

In [ ]:
from src.features import cross_game_feature_cols

cols = cross_game_feature_cols

print("Feature medians (RE human vs CSGO human vs CSGO bots):")
compare_re_csgo = pd.DataFrame({
    "RE_human": games_df[cols].median(),
    "CSGO_human": csgo_games_df[cols].median(),
    "CSGO_stitch": csgo_bots_stitch_df[cols].median(),
    "CSGO_smooth": csgo_bots_smooth_df[cols].median(),
    "CSGO_bezier": csgo_bots_bezier_df[cols].median(),
    "CSGO_vae": csgo_bots_vae_df[cols].median(),
}).round(3)
print(compare_re_csgo)



## Cross-game transfer (CSGO train → RE test)

In [ ]:
from src.features import cross_game_feature_cols
from src.evaluation import train_bot_detector

print("=== CSGO model trained on STITCH bots (7 cross-game features) ===")
csgo_x_model_stitch, csgo_x_acc_stitch = train_bot_detector(
    csgo_games_df, csgo_bots_stitch_df, cross_game_feature_cols, name="CSGO stitch"
)
print()
print("=== CSGO model trained on SMOOTH bots (7 cross-game features) ===")
csgo_x_model_smooth, csgo_x_acc_smooth = train_bot_detector(
    csgo_games_df, csgo_bots_smooth_df, cross_game_feature_cols, name="CSGO smooth"
)
print()
print("=== CSGO model trained on BEZIER bots (7 cross-game features) ===")
csgo_x_model_bezier, csgo_x_acc_bezier = train_bot_detector(
    csgo_games_df, csgo_bots_bezier_df, cross_game_feature_cols, name="CSGO bezier"
)

print()
print("=== CSGO model trained on VAE bots (7 cross-game features) ===")
csgo_x_model_vae, csgo_x_acc_vae = train_bot_detector(
    csgo_games_df, csgo_bots_vae_df, cross_game_feature_cols, name="CSGO vae"
)


## Zero-shot diagnose (raw features, CSGO → RE)

In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import cross_game_feature_cols

print("=== Raw features: true zero-shot CSGO -> RE ===")
print("(threshold from RE humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", csgo_x_model_stitch, games_df, bots_stitch_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "smooth", csgo_x_model_smooth, games_df, bots_smooth_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "bezier", csgo_x_model_bezier, games_df, bots_bezier_df,
    cross_game_feature_cols, title_suffix="raw",
)

print()
_ = diagnose_cross_game(
    "vae", csgo_x_model_vae, games_df, bots_vae_df,
    cross_game_feature_cols, title_suffix="raw",
)


## Scale-invariant train (CSGO) — for RE transfer



In [ ]:
from src.evaluation import train_bot_detector
from src.features import to_scale_invariant, SCALE_INVARIANT_COLS
from src.config import RNG_SEED

csgo_si_human_tr = to_scale_invariant(csgo_games_df)
csgo_si_stitch_tr = to_scale_invariant(csgo_bots_stitch_df)
csgo_si_smooth_tr = to_scale_invariant(csgo_bots_smooth_df)
csgo_si_bezier_tr = to_scale_invariant(csgo_bots_bezier_df)
csgo_si_vae_tr = to_scale_invariant(csgo_bots_vae_df)

print("=== Train on CSGO (scale-invariant features) ===")
m_csgo_si_stitch, _ = train_bot_detector(
    csgo_si_human_tr, csgo_si_stitch_tr, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="CSGO scale-invariant stitch",
    show_feature_importance=False,
)
print()
m_csgo_si_smooth, _ = train_bot_detector(
    csgo_si_human_tr, csgo_si_smooth_tr, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="CSGO scale-invariant smooth",
    show_feature_importance=False,
)
print()
m_csgo_si_bezier, _ = train_bot_detector(
    csgo_si_human_tr, csgo_si_bezier_tr, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="CSGO scale-invariant bezier",
    show_feature_importance=False,
)

print()
m_csgo_si_vae, _ = train_bot_detector(
    csgo_si_human_tr, csgo_si_vae_tr, SCALE_INVARIANT_COLS,
    random_state=RNG_SEED, name="CSGO scale-invariant vae",
    show_feature_importance=False,
)


## Zero-shot diagnose (scale-invariant features, CSGO → RE)



In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import to_scale_invariant, SCALE_INVARIANT_COLS

re_si_human = to_scale_invariant(games_df)
re_si_stitch = to_scale_invariant(bots_stitch_df)
re_si_smooth = to_scale_invariant(bots_smooth_df)
re_si_bezier = to_scale_invariant(bots_bezier_df)
re_si_vae = to_scale_invariant(bots_vae_df)

print("=== Scale-invariant: true zero-shot CSGO -> RE ===")
print("(threshold from RE humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", m_csgo_si_stitch, re_si_human, re_si_stitch,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)
print()
_ = diagnose_cross_game(
    "smooth", m_csgo_si_smooth, re_si_human, re_si_smooth,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)
print()
_ = diagnose_cross_game(
    "bezier", m_csgo_si_bezier, re_si_human, re_si_bezier,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)

print()
_ = diagnose_cross_game(
    "vae", m_csgo_si_vae, re_si_human, re_si_vae,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)


## Feature scale comparison (CSGO vs RE)

In [ ]:
from src.features import cross_game_feature_cols

cols = cross_game_feature_cols

print("Feature medians (CSGO human vs RE human vs RE bots):")
compare_csgo_re = pd.DataFrame({
    "CSGO_human": csgo_games_df[cols].median(),
    "RE_human": games_df[cols].median(),
    "RE_stitch": bots_stitch_df[cols].median(),
    "RE_smooth": bots_smooth_df[cols].median(),
    "RE_bezier": bots_bezier_df[cols].median(),
    "RE_vae": bots_vae_df[cols].median(),
}).round(3)
print(compare_csgo_re)

